# OCR Head-to-Head — `gemma4` vs `deepseek-ocr`\n\nRuns the same receipts through both vision-capable local models with **identical prompts**, so the comparison isn't biased by model-specific prompt conventions:\n\n- `deepseek-ocr:latest` — 3.3B, OCR-specialist (`vision` only)\n- `gemma4:latest` — 8.0B, general-purpose VLM (`vision`, `tools`, `thinking`, `audio`)\n\nTwo tests: (1) raw verbatim transcription, (2) structured JSON extraction. Also includes `deepseek-ocr`'s native `\"Free OCR.\"` mode as a bonus, since that's its specialized convention rather than a neutral prompt.\n\nMake sure the Ollama server is running (`ollama serve`) before executing cells.

In [1]:
import base64
import json
import time
from pathlib import Path

import requests

BASE_URL = "http://localhost:11434"
MODELS = {"deepseek-ocr": "deepseek-ocr:latest", "gemma4": "gemma4:latest"}

RECEIPTS_DIR = Path("receipts")
receipt_paths = sorted(RECEIPTS_DIR.glob("*.[jp][pn]g"))
[p.name for p in receipt_paths]

['1*N9w_Ck211Lo22lYbTd14aQ.jpg',
 'FE1DgHAWYAAwMgV.jpg',
 'Fake-Hotel-Receipt-Template.jpg',
 'fast-food-receipt-for-ice-cream-frozen-yogurt-custard-store1.png',
 'fast-food-restaurant-template-with-itemized-food-and-tax.png']

In [2]:
def image_to_b64(path: Path) -> str:
    return base64.b64encode(path.read_bytes()).decode()


def run_vision_prompt(model: str, image_path: Path, prompt: str) -> tuple[str, float]:
    start = time.monotonic()
    resp = requests.post(
        f"{BASE_URL}/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "images": [image_to_b64(image_path)],
            "stream": False,
        },
    )
    resp.raise_for_status()
    elapsed = time.monotonic() - start
    return resp.json()["response"], elapsed

## Test 1 — verbatim transcription (neutral prompt)

Same instruction for both models, no OCR-specific conventions.

In [3]:
TRANSCRIBE_PROMPT = (
    "Transcribe all the text visible in this image exactly as it appears, "
    "preserving line breaks. Output only the transcribed text, no commentary."
)

transcriptions = {}
for path in receipt_paths:
    transcriptions[path.name] = {}
    print(f"########## {path.name} ##########")
    for label, model in MODELS.items():
        text, elapsed = run_vision_prompt(model, path, TRANSCRIBE_PROMPT)
        transcriptions[path.name][label] = text
        print(f"--- {label} ({elapsed:.1f}s) ---")
        print(text)
        print()

########## 1*N9w_Ck211Lo22lYbTd14aQ.jpg ##########


--- deepseek-ocr (5.2s) ---




--- gemma4 (8.2s) ---
EPIC STEAKHOUSE
369 THE EMBARCADERO
SAN FRANCISCO, CA 94105

RECEIPT

2 FILET MIGNON $98.00
1 RIB EYE $52.00
1 CAESAR SALAD $14.50
1 CREAMED SPINACH $11.00
1 BAKED POTATO $9.50
$185.00

SUBTOTAL 17.02
TAX $75.00
TIP TOTAL $277.02

########## FE1DgHAWYAAwMgV.jpg ##########


--- deepseek-ocr (5.1s) ---




--- gemma4 (21.8s) ---
BEST DATE NIGHT LOCATION IN TOWN
PIZZA PLACE NY
STATEN ISLAND
NY
10086
CASHIER: KRIS J
CUSTOMER: PETE AND KIM K
PURCHASE:
GOLD PLATED CARAVIA $500.00
GUCCHI GRAPES     $95.00
POOCH PIPER       $1,420.00
KYLIE SALMON      $800.00
KYLIE KARLE       $180.00
STATE TAX  $189.66
RICH TAX   $68.00 TAX
TOTAL     $847.28
PAYMENT METHOD: CREDIT CARD
TRANSACTION #1365982614 -001
DATE: 03/11/2021 3:54:58 PM
THANK YOU

########## Fake-Hotel-Receipt-Template.jpg ##########


...

--- gemma4 (22.2s) ---
Merchant ID:
Term ID:

SALE

VISA
Entry Method: Swiped

Approved Online
09/22/2014

Batch #: 011103
07:50:07

Acct # 33458
Inst Inv: 174558
Appr Code: 690847G

Folio No.
A/R Number
Group Code
Hilton Honors
Level
Invoice No.

Visa
Transaction #: R562/447
Card Type
Entry:
Invoice #:
Total USD $903.28

CUSTOMER COPY
09-18-14 *Accommodation
09-18-14 Lodging Tax
09-18-14 City Tax
09-19-14 *Accommodation
09-19-14 Lodging Tax
09-19-14 City Tax

########## fast-food-receipt-for-ice-cream-frozen-yogurt-custard-store1.png ##########


--- deepseek-ocr (5.4s) ---




--- gemma4 (24.8s) ---
ICE Cream - Frozen Yogurt
23223, DAVIS AVE
APEX, SC, 8822
888 888 8888
Serving All-Natural Frozen Custard

ORDER: 37

1 Regular Concrete $ 5.50
2 Special Oreo Concrete $ 13.00
Vanilla Custard $ 0.75
Wainuts $ 0.49
Pecans $ 0.25
Caramel $ 0.25
$ 0.39
$ 0.76
$ 0.76
$ 0.20

SUBTOTAL $ 22.62
TOTAL $ 22.62

CREDIT CARD AUTH $ 22.62 | 19:57
VISA CREDITS XXXXXXXXXXXX9999
Customer Name Williams
TRANS #:********8221
AUTH #:574W6L9OILTVG3R

Order XBKN2Q5AIND70ZJX
Payment B4254D3A6KAT7D9A

Clover Privacy Policy
https://clover.com/privacy

########## fast-food-restaurant-template-with-itemized-food-and-tax.png ##########


--- deepseek-ocr (6.7s) ---
 If there are any errors or omissions, please let me know.
```
Fish & Chips Fast Foods
2334, Fish and Chips Street
New Hill, SC, 34566-454646
888-888-8888

Order: 454

Host: Meggan F
12-01-2020 12:16 PM

Qty Item Price
2 Fish Burger € 25.98
1 Fish & Chips € 8.99
2 Soft Drink € 3.98

VISA 2277 Sale

Subtotal € 38.95
Tax € 1.17
SalesTax € 1.17
Total: € 41.29

Transaction Type: Sale
Authorization: Approved
Payment Code: 86180556228453
Payment ID: 238642613672835
Card Reader: Swiped/Chip

+ Tip: _______________

=Total: _______________

X _________________________

Customer Copy
Thanks for visiting
Fish & Chips Fast Foods



--- gemma4 (8.9s) ---
Fish & Chips Fast Foods
234 Oak Street
New Mill, SC 34566-4546
888-888-8888

Order: 454

QTY Item Price
2 Fish Burger € 25.98
1 Fish & Chips € 8.99
2 Soft Drink € 3.98

VISA 2277 Sale
Subtotal € 38.95
Tax € 1.17
Total: € 41.29

Transaction Type: Sale
Authorization: Approved
Payment Code: 8618055228453
Payment ID: 238642613672835
Card Reader: Swiped/Chip

+ Tip: _____________
=Total: __________________

Customer Copy
Thanks for visiting
Fish & Chips Fast Foods



### Bonus — `deepseek-ocr` native prompt

`deepseek-ocr` recognizes a special `"Free OCR."` instruction as its dedicated transcription mode — worth checking whether it outperforms the neutral prompt above.

In [4]:
for path in receipt_paths:
    text, elapsed = run_vision_prompt(MODELS["deepseek-ocr"], path, "Free OCR.")
    transcriptions[path.name]["deepseek-ocr (native prompt)"] = text
    print(f"=== {path.name} ({elapsed:.1f}s) ===")
    print(text)
    print()

=== 1*N9w_Ck211Lo22lYbTd14aQ.jpg (8.0s) ===
# EPIC STEAKHOUSE

369 The Embarcadero  
SAN FRANCISCO, CA 94105  

**RECEIPT**

2 FILET MIGNON    $98.00  
1 RIB EYE    $52.00  
1 CAESAR SALAD    $14.50  
1 CREAMED SPINACH   $9.50  
1 BAKED POTATO    $185.00  

**SUBTOTAL**    17.02  
**TAX**    $75.00  
**TIP**    **$277.02**

**TOTAL**    **$277.02**

---

# EPIC STEAKHOUSE

369 The Embarcadero  
SAN FRANCISCO, CA 94105  

**RECEIPT**

2 FILET MIGNON    $98.00  
1 RIB EYE    $52.00  
1 CAESAR SALAD    $14.50  
1 CREAMED SPINACH   $11.00  
1 BAKED POTATO    $185.00  

**SUBTOTAL**    17.02  
**TAX**    $75.00  
**TIP**    **$277.02**

**TOTAL**    **$277.02**

---

# CHAUNCEY'S STEAK RESTAURANT

123 Main Street,  
Montgomery, AL 36104  

(334) 555-1234  

**Server:** John D.,  
Table: 5  
**Receipt #:** 20250312-703  
**Date:** March 12, 2025  

2 Ribeye Steak    $84.00  
2 House Salad    $20.00  
2 Soft Drinks    $10.00  

**Food**  
Tax (10%)    $114.00  
Tip    $11.40  
**Total Due:** 

=== FE1DgHAWYAAwMgV.jpg (2.6s) ===
BEST DATE NIGHT LOCATION IN TOWN

PIZZA PLACE NY  
STATEN ISLAND  
NY  
10306  

---  
CASHIER: KRIS J  
CUSTOMER: PETE AND KIM K  

---  
PURCHASE:  

GOLD PLATED CAVIAR    $500.00  
GUCCI GRAPES    $555.00  
POOSH PIZZA    $740.00  
SKIMS SALMON    $800.00  
KYLIE KALE    $100.00  
STATE TAX +5.00% TAX:   $139.55  
RICH TAX +20.0% TAX:     $550.20  

---  
TOTAL: $3400.75  

PAYMENT METHOD: CREDIT CARD  
TRANSACTION #1685902614 -001  
DATE: 03/11/2021 3:54:50 PM  

THANK YOU



=== Fake-Hotel-Receipt-Template.jpg (2.3s) ===
Merchant ID:  
Term ID:  

SALE  

VISA  
Entry Method: Swiped  

Approved Online  
09/22/2014  

Batch#: 011103  
07:50:07  

Inv#: 174558  
Appr Code: 6908476  

Transaction #: RS662/447  
Card Type: USSA  
Acc:  
Entry: 33458  
Invoice #: 825779  
Total USD $903.28  

CUSTOMER COPY  

09-18-14 *Accommodation  
09-18-14 Lodging Tax  
09-18-14 City Tax  
09-19-14 *Accommodation  
09-19-14 Lodging Tax  
09-19-14 City Tax



=== fast-food-receipt-for-ice-cream-frozen-yogurt-custard-store1.png (3.6s) ===
ICE Cream - Frozen Yogurt  
23223, DAVIS AVE  
APEX, SC, 32322  
888 888 8888  

Serving All-Natural Frozen Custard  

**ORDER:** 37  
05/24/2021 19:55  
Transaction  6896571  

| Item | Description    | Price |
|---|---|---|
| 1    | Regular Concrete    | $ 5.50 |
|    | Vanilla Custard    | $ 0.75 |
|    | Walnuts    | $ 0.49 |
|    | Pecans    | $ 0.39 |
|    | Caramel    | $ 0.25 |

| 2    | Special Oreo Concrete   | $ 13.00|
|    | Caramel    | $ 0.50 |
|    | Pecans    | $ 0.78 |
|    | Vanilla Custard    | $ 0.76 |
|    | Chocs    | $ 0.20 |

**SUBTOTAL**  
$ 22.62  

**TOTAL**  
$ 22.62  

CREDIT CARD AUTH  
$ 22.62  

05/24/2021 19:57  
$ 22.62 | EMV  
VISA CREDIT XXXXXXXXXXXXX9999  
Customer Name  
Reference ID 870357112634 | Auth  
ID UBMMG  

MID : *******8221  
AID : 574W6L9OITLVGW3R  

14315293111489220000  

Order XBKN2Q5AIND70ZJX  
Payment B4254D39A67KATD9  

Clover Privacy Policy  
https://

=== fast-food-restaurant-template-with-itemized-food-and-tax.png (2.8s) ===
Fish & Chips Fast Foods  
2334, Fish and Chips Street  
New Hill, SC, 34566-454646  
888-888-8888  

**Order: 454**

Host: Meggan F  
12-01-2020    12:16 PM  

| Qty | Item                | Price |
|-----|---------------------|-------|
| 2   | Fish Burger         | € 25.98 |
| 1   | Fish & Chips        | € 8.99  |
| 2   | Soft Drink          | € 3.98  |

VISA 2277    Sale  

Subtotal     € 38.95  
Tax            € 1.17  
SalesTax      € 1.17  
Total:       € 41.29  

Transaction Type: Sale  
Authorization: Approved  
Payment Code: 86180556228453  
Payment ID: 238642613672835  
Card Reader: Swiped/Chip  

+ Tip: ______________

=Total: ______________

X _________________________

Customer Copy  
Thanks for visiting  
Fish & Chips Fast Foods



## Test 2 — structured JSON extraction

Same extraction prompt for both models. This tests instruction-following (valid JSON, correct shape) on top of raw text recognition.

In [5]:
EXTRACTION_PROMPT = (
    "Extract this receipt as JSON with keys: "
    '"merchant", "date", "items" (list of {"name", "price"}), "total". '
    "Respond with only the JSON object, no commentary."
)

def parse_json_response(raw: str):
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        if cleaned.startswith("json"):
            cleaned = cleaned[4:]
    try:
        return json.loads(cleaned.strip())
    except json.JSONDecodeError:
        return None

extractions = {}
for path in receipt_paths:
    extractions[path.name] = {}
    print(f"########## {path.name} ##########")
    for label, model in MODELS.items():
        raw, elapsed = run_vision_prompt(model, path, EXTRACTION_PROMPT)
        parsed = parse_json_response(raw)
        extractions[path.name][label] = {"raw": raw, "parsed": parsed}
        status = "valid JSON" if parsed is not None else "FAILED to parse"
        print(f"--- {label} ({elapsed:.1f}s, {status}) ---")
        print(json.dumps(parsed, indent=2) if parsed is not None else raw)
        print()

########## 1*N9w_Ck211Lo22lYbTd14aQ.jpg ##########


--- deepseek-ocr (1.2s, FAILED to parse) ---
The given text is a receipts from two different restaurants, Epic Steakhouse and Chauncey's Steak Restaurant. The receipt lists various food items and their prices along with the total amount due. The restaurant charges $98 for filet mignon, $52 for rib eye steak, and $14 for caesar salad. They also offer a baked potato for $9.50 and a subtotal of $185.00. The Chauncey's Steak Restaurant charges $123 for an item from the main street menu, with prices ranging from $334 to $555. The receipt includes the server's name, John D., table number 5, and date of March 12, 2025. It lists items such as ribeye steak, house salad, soft drinks, food tax, tip, and total due. The restaurant charges a total of $132.45.



--- gemma4 (21.9s, valid JSON) ---
{
  "merchant": "CHAUNCEY'S STEAK RESTAURANT",
  "date": "March 12, 2025",
  "items": [
    {
      "name": "Ribeye Steak",
      "price": "$84.00"
    },
    {
      "name": "House Salad",
      "price": "$20.00"
    },
    {
      "name": "Soft Drinks",
      "price": "$10.00"
    }
  ],
  "total": "$132.45"
}

########## FE1DgHAWYAAwMgV.jpg ##########


--- deepseek-ocr (5.1s, FAILED to parse) ---




--- gemma4 (30.4s, valid JSON) ---
{
  "merchant": "BEST DATE NIGHT LOCATION IN TOWN",
  "date": "03/11/2021",
  "items": [
    {
      "name": "GOLD PLATED CARAVIA",
      "price": 500.0
    },
    {
      "name": "GUCCI GRAPES",
      "price": 595.0
    },
    {
      "name": "BOSS PARFAUM",
      "price": 500.0
    },
    {
      "name": "SKIING SALMON",
      "price": 800.0
    },
    {
      "name": "KYLIE KALIE",
      "price": 180.0
    }
  ],
  "total": 3488.75
}

########## Fake-Hotel-Receipt-Template.jpg ##########


...

--- gemma4 (28.6s, valid JSON) ---
{
  "merchant": "Hilton Honors",
  "date": "09/22/2014",
  "items": [
    {
      "name": "*Accommodation (09-18-14)",
      "price": null
    },
    {
      "name": "Lodging Tax (09-18-14)",
      "price": null
    },
    {
      "name": "City Tax (09-18-14)",
      "price": null
    },
    {
      "name": "*Accommodation (09-19-14)",
      "price": null
    },
    {
      "name": "Lodging Tax (09-19-14)",
      "price": null
    },
    {
      "name": "City Tax (09-19-14)",
      "price": null
    }
  ],
  "total": "USD 903.28"
}

########## fast-food-receipt-for-ice-cream-frozen-yogurt-custard-store1.png ##########


--- deepseek-ocr (6.6s, FAILED to parse) ---
The provided text is a screenshot of an online purchase transaction made on May 24, 2021. The order number is 37 and it was placed at 19:55. The items purchased include Regular Concrete (5.50), Vanilla Custard (0.75), Walnuts (0.49), Pecans (0.39), Caramel (0.25), Special Oreo Concrete (13.00), Caramel (0.50), Pecans (0.78), Vanilla Custard (0.76), and Chocs (0.20). The total cost of the order is $22.62, which includes a credit card charge for EMV Visa Credit XXXXXXXXXXXXX9999 with customer name, reference ID 870357112634, and auth ID UBMMG. The transaction also shows an AID of 574W6L9OITLVGW3R.



--- gemma4 (21.6s, valid JSON) ---
{
  "merchant": "ICE Cream - Frozen Yogurt",
  "date": "",
  "items": [
    {
      "name": "Regular Concrete",
      "price": 5.5
    },
    {
      "name": "Special Oreo Concrete",
      "price": 13.0
    }
  ],
  "total": 22.62
}

########## fast-food-restaurant-template-with-itemized-food-and-tax.png ##########


--- deepseek-ocr (6.4s, valid JSON) ---
{
  "merchant": "Fish & Chips Fast Foods",
  "date": "12-01-2020",
  "items": [
    {
      "name": "Fish Burger",
      "price": 25.98
    },
    {
      "name": "Fish & Chips",
      "price": 8.99
    },
    {
      "name": "Soft Drink",
      "price": 3.98
    }
  ],
  "total": 41.29,
  "transaction_type": "Sale",
  "authorization": "Approved",
  "payment_code": "86180556228453",
  "payment_id": "238642613672835",
  "card_reader": "Swiped/Chip"
}



--- gemma4 (25.3s, valid JSON) ---
{
  "merchant": "Fish & Chips Fast Foods",
  "date": "12-01-2020",
  "items": [
    {
      "name": "Fish Burger",
      "price": "25.98"
    },
    {
      "name": "Fish & Chips",
      "price": "8.99"
    },
    {
      "name": "Soft Drink",
      "price": "3.98"
    }
  ],
  "total": "\u20ac 40.12"
}



## Scoreboard

Simple tally: how often each model returned parseable JSON for the structured-extraction test.

In [6]:
for label in MODELS:
    valid = sum(1 for r in extractions.values() if r[label]["parsed"] is not None)
    print(f"{label:<12} valid JSON: {valid}/{len(receipt_paths)}")

deepseek-ocr valid JSON: 1/5
gemma4       valid JSON: 5/5


## Notes

- `deepseek-ocr` is a narrow OCR specialist: strong at raw glyph transcription, but with a neutral (non-native) prompt or a JSON-shape instruction it tends to drift into summarizing/paraphrasing instead of transcribing, and often breaks JSON formatting.
- `gemma4` is a general-purpose VLM: more reliable at following the JSON schema, but as a reasoning model it sometimes infers or normalizes values (e.g. computing a subtotal, reformatting a date) rather than transcribing verbatim — worth checking against the raw transcription in `transcriptions` when accuracy matters.
- For this task on 8-bit consumer receipts, treat `deepseek-ocr` + its native `"Free OCR."` prompt as the source of truth for raw text, and `gemma4` as better suited to reshaping/interpreting that text into structured fields.